In [1]:
import math
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import torch
from torch.utils.data import Dataset
from transformers import DebertaV2Tokenizer

class ORDataset(Dataset):
    """用于有序分类评分任务的WSD数据集"""
    
    def __init__(self, json_path, tokenizer, max_length=512, mode='train'):
        """
        Args:
            json_path (str): 数据文件路径.
            tokenizer (DebertaV2Tokenizer): 分词器.
            max_length (int): 最大序列长度.
            mode (str): 数据集模式 ('train', 'eval', 'predict').
        """
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.mode = mode
        self.samples = []
        
        # 1. 读取JSON
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # 2. 构造样本
        for key, item in data.items():
            
            # --- 文本信息 ---
            homonym = item["homonym"]
            definition = item["judged_meaning"]
            example = item["example_sentence"]
            context = f"{item['precontext']} {item['sentence']} {item['ending']}"
            
            # --- 标签信息 ---
            target_avg = item.get("average", 0.0)
            # target_stdev = item.get("stddev", 0.0)
            
            # target_score = round(target_avg)
            target_score = math.floor(target_avg + 0.5)
            target_score = max(1, min(5, target_score)) # 确保在 1-5 范围内

            
            self.samples.append({
                "json_key": key,
                "homonym": homonym,
                "definition": definition,
                "example": example,
                "context": context,
                # 0 到 4 的整数标签
                "ordinal_label": target_score - 1, 
                # "target_avg": target_avg,
                # "target_stdev": target_stdev,
                
                "sample_id": item['sample_id']
            })
            
        print(f"✅ 创建了 {len(self.samples)} 个有序分类样本 (Mode: {self.mode})")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # 1. 构造输入文本
        text_parts = (
            f"homonym：{sample['homonym']}",
            f"Definition:{sample['definition']}",
            f"Example:{sample['example']}",
            f"Context:{sample['context']}"
        )
        text = self.tokenizer.sep_token.join(text_parts)
        
        # 2. Tokenize
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt" 
        )
        
        # 移除batch维度
        encoding = {k: v.squeeze(0) for k, v in encoding.items()}
        
        # 移除 token_type_ids
        if "token_type_ids" in encoding:
            del encoding["token_type_ids"]
            
        # 3. 标签分配：根据模式返回不同的标签
        
        # 无论哪种模式，我们都需要 ID 
        # ID 作为字符串返回，需要 DataLoader/DataCollator 特殊处理
        encoding["id"] = sample["json_key"] 
        
        if self.mode != 'predict':
            encoding["labels"] = torch.tensor(sample["ordinal_label"], dtype=torch.long)
        else:  # 'predict' 模式
            pass
        
        # if self.mode == 'eval':
        #     encoding["avg_raw"] = torch.tensor(sample["target_avg"], dtype=torch.float32) 
        #     encoding["stdev_raw"] = torch.tensor(sample["target_stdev"], dtype=torch.float32)
        
        # encoding["labels"] = labels_value
        return encoding
    
    

In [2]:
import numpy as np
import torch
from scipy.stats import spearmanr
from typing import Dict, Any

def compute_metrics_simplified(p: Dict[str, Any]) -> Dict[str, float]:
    
    # 1. 提取预测分数 (Logits -> 0-4 整数)
    # p.predictions 是模型输出的 Logits (K-1 Logits, 形状 [N, 4])
    logits = torch.tensor(p.predictions)
    probs = torch.sigmoid(logits) 
    
    # K-1 有序分类模型转为 0-4 标签：1 + 跨越 0.5 阈值的数量 - 1
    # 数量统计范围是 [0, 4]
    predicted_labels_np = (torch.sum(probs > 0.5, dim=1)).numpy() # [N] (0-4 整数)

    # 2. 提取真实标签 (这是 ORDataset 中 floor/round 后的 0-4 整数)
    gold_labels_np = p.label_ids # [N] (0-4 整数)

    # --- A. Spearman Correlation (基于 0-4 整数标签) ---
    corr, _ = spearmanr(predicted_labels_np, gold_labels_np)
    
    # --- B. 💥 核心检查：Accuracy within 1 (使用 labels +/- 1) ---
    
    # 1. 计算绝对差值：|预测 - 真实|
    abs_diff = np.abs(predicted_labels_np - gold_labels_np)
    
    # 2. 命中条件：检查绝对差值是否小于等于 1
    # (即 abs_diff == 0, abs_diff == 1 均视为正确)
    is_correct_within_1 = (abs_diff <= 1)
    
    # 3. 计算命中数量和准确率
    acc_within_1 = np.sum(is_correct_within_1) / len(predicted_labels_np)
    
    # --- C. 严格精度 (Accuracy == 0) ---
    # 检查绝对差值是否等于 0
    strict_accuracy = np.sum(abs_diff == 0) / len(predicted_labels_np)


    return {
        "spearman_rho": float(corr),
        "acc_within_1": float(acc_within_1),
        "strict_accuracy": float(strict_accuracy),
        # "eval_loss": p.metrics.get("eval_loss"),
    }

In [3]:
import torch.nn as nn
from transformers import DebertaV2Model, DebertaV2PreTrainedModel
import torch

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import DebertaV2Model, DebertaV2PreTrainedModel


class DebertaV2ForOrdinalRegression(DebertaV2PreTrainedModel):
    
    def __init__(self, config):
        super().__init__(config)
        self.num_classes = 5
        self.deberta = DebertaV2Model(config)
        
        self.regressor = nn.Sequential(
            nn.Linear(config.hidden_size, config.hidden_size),
            nn.GELU(),
            # nn.LayerNorm(config.hidden_size),
            nn.Linear(config.hidden_size, self.num_classes - 1)    # 4
        )


    # ----------------------------------------------------------------------

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        
        # 编码
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
        
        X = outputs[0][:, 0, :] 
        
        # 1. 预测潜变量 Y*
        logits = self.regressor(X) # 形状: [batch_size, 4]
        
        # mean = logits.mean(dim=0)
        # std = logits.std(dim=0)
        # print(f"Logits 均值: {mean}")
        # print(f"Logits 标差: {std}")
        
        # 2. 损失计算
        if labels is not None:
            
            labels_float = labels.float().unsqueeze(1)
            
            # 构造阈值张量: [1, 2, 3, 4]
            thresholds = torch.arange(1, self.num_classes, dtype=torch.float, device=labels.device).unsqueeze(0)
            
            # T 是二元目标标签，形状 [B, 4]
            target_labels = (labels_float > thresholds).float()
            
            # 3. 计算 BCE 损失
            # logits: [B, 4], target_labels: [B, 4]
            loss = F.binary_cross_entropy_with_logits(logits, target_labels) # 使用功能强大的 F.binary_cross_entropy_with_logits
            # loss_bce = F.binary_cross_entropy_with_logits(logits, target_labels)
            # std_devs = logits.std(dim=0)
            # loss_stddev = -std_devs.mean()
            # lambda_reg = 0.01
            # loss = loss_bce + lambda_reg * loss_stddev
        else:
            loss = None
             
        # 这里的 logits 是 K-1 个 Logits，而不是最终分数
        return (loss, logits) if loss is not None else logits
    
    
    
    def predict_score(self, input_ids, attention_mask):
        # 获取 logits (形状: [B, 4])
        # 确保 predict_score 是类方法
        with torch.no_grad():
            logits = self.forward(input_ids, attention_mask)[1] if isinstance(self.forward(input_ids, attention_mask), tuple) else self.forward(input_ids, attention_mask)
        
        # 1. 转换为概率 P(Y > j)
        probs = torch.sigmoid(logits) # 形状: [B, 4]

        # 2. 计算预测分数
        # 分数 = 1 + 所有概率 > 0.5 的数量之和
        predicted_score = torch.sum(probs > 0.5, dim=1) + 1 
        
        return predicted_score.long()

2025-12-12 19:28:15.166870: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765567695.377772      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765567695.441740      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [4]:
import os
os.environ["WANDB_MODE"] = "disabled"

from transformers import DebertaV2Tokenizer, DebertaV2Config, Trainer, TrainingArguments

# from model import DebertaV2ForOrdinalRegression
# from data_load import ORDataset



# --- 配置 ---
MODEL_NAME = "/kaggle/input/semeval/deberta-v3-large" 
TRAIN_JSON_PATH = "/kaggle/input/semeval/data/train.json" 
TEST_JSON_PATH = "/kaggle/input/semeval/data/dev.json" # 使用 dev.json 进行验证
OUTPUT_DIR = "/kaggle/working/output_ordinal_regression"

# ----------------------------------------------------------------------
# 2. 主训练流程
# ----------------------------------------------------------------------
if __name__ == "__main__":
    
    # 加载 Tokenizer 和配置
    tokenizer = DebertaV2Tokenizer.from_pretrained(MODEL_NAME)
    config = DebertaV2Config.from_pretrained(MODEL_NAME)
    
    # 实例化序数回归模型
    # 注意：这里不再需要 num_labels=1，因为模型内部处理了输出维度
    # model = DebertaV2ForOrdinalRegression.from_pretrained(config=config)
    model = DebertaV2ForOrdinalRegression.from_pretrained(MODEL_NAME, config=config)

    # 实例化数据集
    train_dataset = ORDataset(
        json_path=TRAIN_JSON_PATH,
        tokenizer=tokenizer,
        mode='train',
        max_length=256
        )
    eval_dataset = ORDataset(
        json_path=TEST_JSON_PATH,
        tokenizer=tokenizer,
        max_length=256,
        mode='eval')

    # 训练参数配置
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=10,                       # 增加 epochs
        per_device_train_batch_size=8,
        # gradient_accumulation_steps=4,

        save_total_limit=1,
        eval_strategy="epoch",  
        save_strategy="epoch",
        load_best_model_at_end=True,  
        metric_for_best_model="spearman_rho", 
        greater_is_better=True,
        
        warmup_steps=500,                       
        weight_decay=0.01,                      
        logging_dir='./logs_ordinal',       
        logging_steps=50,                       
        learning_rate=2e-5,                     
        fp16=True,                              
        seed=42,
        # remove_unused_columns=False,                   
    )

    # 实例化 Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics_simplified,
    )

    # 启动训练
    print("***** 开始微调 DeBERTaV2 序数回归模型 *****")
    trainer.train()

    # 训练结束后，保存最终模型
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"训练完成，最终最佳模型已保存至 {OUTPUT_DIR}")

Some weights of DebertaV2ForOrdinalRegression were not initialized from the model checkpoint at /kaggle/input/semeval/deberta-v3-large and are newly initialized: ['regressor.0.bias', 'regressor.0.weight', 'regressor.2.bias', 'regressor.2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_47/2913836972.py:69: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


✅ 创建了 2280 个有序分类样本 (Mode: train)
✅ 创建了 588 个有序分类样本 (Mode: eval)


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

***** 开始微调 DeBERTaV2 序数回归模型 *****


Epoch,Training Loss,Validation Loss,Spearman Rho,Acc Within 1,Strict Accuracy
1,0.444400,0.443262,nan,0.588435,0.224490
2,0.446000,0.462795,0.249413,0.479592,0.154762
3,0.346600,0.372528,0.492996,0.695578,0.267007
4,0.267500,0.474225,0.528944,0.785714,0.314626
5,0.202900,0.503480,0.480898,0.729592,0.312925
6,0.145900,0.566948,0.496055,0.710884,0.265306
7,0.110900,0.667232,0.495632,0.702381,0.253401
8,0.078000,0.792637,0.482275,0.750000,0.297619
9,0.044600,0.835128,0.495139,0.724490,0.290816
10,0.022200,0.878075,0.490102,0.727891,0.282313


/tmp/ipykernel_47/4078969949.py:21: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, _ = spearmanr(predicted_labels_np, gold_labels_np)
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:3177: RuntimeWarning: invalid value encountered in greater
  if operator(metric_value, self.state.best_metric):


训练完成，最终最佳模型已保存至 /kaggle/working/output_ordinal_regression


In [5]:
import os
import json
import torch
import numpy as np
from torch.utils.data import DataLoader, SequentialSampler
from tqdm import tqdm
# ⚠️ 假设 ORDataset, tokenizer, model, model.device 已在外部定义和加载

# ----------------------------------------------------------------------
# 路径和配置 (保持不变)
# ----------------------------------------------------------------------
TEST_JSON_PATH = "/kaggle/input/semeval/data/dev.json" 
INFERENCE_BATCH_SIZE = 32
OUTPUT_DIR = "/kaggle/working/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_JSONL_PATH = os.path.join(OUTPUT_DIR, "predictions_scale.jsonl")

# 确定设备
device = model.device 
model.eval() 

print(f"Model is on: {device}")
print(f"Using Test/Inference file: {TEST_JSON_PATH}")

# ----------------------------------------------------------------------
# 1. 实例化推理数据集和 DataLoader (保持不变)
# ----------------------------------------------------------------------
dev_dataset = ORDataset(
    json_path=TEST_JSON_PATH, 
    tokenizer=tokenizer,
    max_length=256,
    mode='predict',
)

dev_dataloader = DataLoader(
    dev_dataset,
    sampler=SequentialSampler(dev_dataset),
    batch_size=INFERENCE_BATCH_SIZE,
)

# ----------------------------------------------------------------------
# 2. 运行推理循环 (核心修改)
# ----------------------------------------------------------------------
all_results = []
# ❌ 不再需要 score_tensor，因为不再计算期望值 (Expected Value)
# score_tensor = torch.tensor([1, 2, 3, 4, 5], dtype=torch.float32, device=device).unsqueeze(0)

print("\n***** 开始批量 K-1 二分类推理 *****")

with torch.no_grad():
    for batch in tqdm(dev_dataloader, desc="Inferencing"):
        
        # 提取输入和 ID
        ids = batch.pop("id") 
        
        # 运行模型
        # ✅ 核心调用：直接使用 model.predict_score 方法进行推理
        # predict_score 方法内部会处理 Logits -> Probs -> Sum(Probs > 0.5) + 1
        final_scores = model.predict_score(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device)
        )
        # final_scores 形状: [batch_size]，类型: torch.long (1到5的整数)

        # ❌ 移除复杂的 outputs 处理和期望值计算：
        # if isinstance(outputs, dict):
        #    ...
        # expected_scores = torch.sum(probs * score_tensor, dim=1)
        # final_scores = torch.clamp(expected_scores.round(), 1, 5).int()
        
        # 收集结果
        for json_key, score in zip(ids, final_scores.tolist()):
            all_results.append({
                "id": json_key, 
                "prediction": int(score) # score 已经是 1-5 的整数
            })

print("\nK-1 二分类推理完成。")

# ----------------------------------------------------------------------
# 3. 保存为 JSON Lines (.jsonl) (保持不变)
# ----------------------------------------------------------------------
with open(OUTPUT_JSONL_PATH, 'w', encoding='utf-8') as f:
    for result in all_results:
        f.write(json.dumps(result) + '\n')

print(f"✅ 所有预测结果已保存到 {OUTPUT_JSONL_PATH}")

Model is on: cuda:0
Using Test/Inference file: /kaggle/input/semeval/data/dev.json
✅ 创建了 588 个有序分类样本 (Mode: predict)

***** 开始批量 K-1 二分类推理 *****


Inferencing: 100%|██████████| 19/19 [00:23<00:00,  1.23s/it]


K-1 二分类推理完成。
✅ 所有预测结果已保存到 /kaggle/working/predictions_scale.jsonl


In [6]:
# import os
# import json
# import torch
# import numpy as np
# from torch.utils.data import DataLoader, SequentialSampler
# from tqdm import tqdm
# # 确保 ORDataset, tokenizer, model 已经可用
# INFERENCE_BATCH_SIZE = 32
# TEST_JSON_PATH = "/kaggle/input/semeval/data/dev.json"
# # ----------------------------------------------------------------------
# # A. 辅助函数：自定义 Collate Fn (处理字符串 ID)
# # ----------------------------------------------------------------------
# def inference_collate_fn(batch):
#     """用于推理的 Collate 函数：处理张量和非张量数据 (如 ID)"""
#     ids = [item.pop("id") for item in batch]
#     # 剩余的特征现在只有 input_ids 和 attention_mask (都是张量)
#     tensor_batch = torch.utils.data.dataloader.default_collate(batch)
#     tensor_batch["id"] = ids
#     return tensor_batch


# # ----------------------------------------------------------------------
# # 路径和配置
# # ----------------------------------------------------------------------

# # 确定设备
# # ⚠️ 确保 model 和 tokenizer 在此处被加载，否则 device 引用会失败
# # 假设 model 已经被加载
# device = model.device 
# model.eval() 

# print(f"Model is on: {device}")
# print(f"Using Test/Inference file: {EVAL_JSON_PATH}")

# # ----------------------------------------------------------------------
# # 1. 实例化推理数据集和 DataLoader
# # ----------------------------------------------------------------------
# test_dataset = InferenceDataset(
#     json_path=TEST_JSON_PATH, 
#     tokenizer=tokenizer, 
# )
# test_dataloader = DataLoader(
#     test_dataset,
#     sampler=SequentialSampler(test_dataset),
#     batch_size=INFERENCE_BATCH_SIZE,
#     collate_fn=inference_collate_fn # 必须使用自定义 collate_fn 
# )

# # ----------------------------------------------------------------------
# # 2. 运行推理循环 (保持不变)
# # ----------------------------------------------------------------------
# all_results = []
# score_values = torch.tensor([1, 2, 3, 4, 5], dtype=torch.float32, device=device).unsqueeze(0)

# print("\n***** 开始批量手动推理 *****")

# with torch.no_grad():
#     for batch in tqdm(test_dataloader, desc="Inferencing"):
        
#         ids = batch.pop("id") 
#         input_ids = batch["input_ids"].to(device)
#         attention_mask = batch["attention_mask"].to(device)
        
#         # 运行模型
#         outputs = model(
#             input_ids=input_ids,
#             attention_mask=attention_mask
#         )
        
#         # 💥 必须根据模型forward的最终调整结果来解包！
#         # 如果您采用了我最后一个建议 (Likelihoods放在Y*前面)：
#         likelihoods, y_star = outputs
        
#         # 1. 计算期望值 (Mean Prediction)
#         mean_predictions = (likelihoods * score_values).sum(dim=1) 
        
#         # 2. 四舍五入到最近的整数 (1-5)
#         final_scores_float = mean_predictions.cpu().numpy()
#         final_scores = np.round(final_scores_float).astype(int)
        
#         # 3. 确保分数在 1 到 5 之间
#         final_scores = np.clip(final_scores, 1, 5)
        
#         # 收集结果
#         for json_key, score in zip(ids, final_scores.tolist()):
#             all_results.append({
#                 "id": json_key, 
#                 "prediction": int(score)
#             })

# print("\n手动推理完成。")

# # ----------------------------------------------------------------------
# # 3. 保存为 JSON Lines (.jsonl)
# # ----------------------------------------------------------------------
# with open(OUTPUT_JSONL_PATH, 'w', encoding='utf-8') as f:
#     for result in all_results:
#         f.write(json.dumps(result) + '\n')

# print(f"✅ 所有预测结果已保存到 {OUTPUT_JSONL_PATH}")

In [7]:
# import torch
# import numpy as np
# import json
# from torch.utils.data import DataLoader, Dataset, SequentialSampler
# from transformers import DebertaV2Tokenizer, DebertaV2Config
# from tqdm.auto import tqdm # 假设您使用 tqdm 来显示进度

# # 假设您已将 model.py 中的类（包括 OrdinalRegressionLoss 和 DebertaV2ForOrdinalRegression）导入
# # from model import DebertaV2ForOrdinalRegression 
# # 假设您已将 InferenceDataset 导入或定义在脚本中
# # from data_load import InferenceDataset 


# # --- 配置 ---
# MODEL_PATH = "/kaggle/working/output_ordinal_regression"
# TEST_JSON_PATH = "/kaggle/input/semeval/data/dev.json" # 请替换为您的实际测试数据路径
# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# INFERENCE_BATCH_SIZE = 16 

# # --- 注意：请确保您在 inference.py 中定义了 InferenceDataset 和 inference_collate_fn ---
# # (为了简洁，这里省略了它们的代码，假设您已将它们复制到此脚本或导入)

In [8]:
# def inference_collate_fn(batch):
#     """用于推理的 Collate 函数：处理张量和非张量数据 (如 ID)"""
#     # 提取 ID (字符串)
#     ids = [item.pop("id") for item in batch]
#     # 剩余的特征进行张量批处理
#     tensor_batch = torch.utils.data.dataloader.default_collate(batch)
#     tensor_batch["id"] = ids
#     return tensor_batch

In [9]:
# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# def run_inference():
#     print(f"载入模型组件自: {MODEL_PATH}")
    
#     # 1. 加载 Tokenizer, Config 和模型
#     tokenizer = DebertaV2Tokenizer.from_pretrained(MODEL_PATH)
#     config = DebertaV2Config.from_pretrained(MODEL_PATH)
    
#     # 💥 注意：from_pretrained 会在加载权重时检查模型的 forward 参数，这正是我们之前修复 TypeError 的原因
#     model = DebertaV2ForOrdinalRegression.from_pretrained(MODEL_PATH, config=config)
#     model.to(DEVICE)
#     model.eval() # 切换到评估模式 (关闭 Dropout/BatchNorm)
    
#     # 2. 实例化数据集和 DataLoader
#     test_dataset = InferenceDataset(json_path=TEST_JSON_PATH, tokenizer=tokenizer)
#     test_dataloader = DataLoader(
#         test_dataset,
#         sampler=SequentialSampler(test_dataset),
#         batch_size=INFERENCE_BATCH_SIZE,
#         collate_fn=inference_collate_fn
#     )

#     all_results = []
#     # 类别分数：[1.0, 2.0, 3.0, 4.0, 5.0]，用于计算期望值
#     score_values = torch.tensor([1, 2, 3, 4, 5], dtype=torch.float32, device=DEVICE).unsqueeze(0)

#     print("\n***** 开始推理 *****")

#     with torch.no_grad():
#         for batch in tqdm(test_dataloader, desc="Inferencing"):
            
#             ids = batch.pop("id") 
#             input_ids = batch["input_ids"].to(DEVICE)
#             attention_mask = batch["attention_mask"].to(DEVICE)
            
#             # 运行模型
#             # 💥 模型输出：(Likelihoods, Y*)，因为我们在 forward 中做了最后的调整
#             outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            
#             # 3. 解析结果 (Likelihoods 是第一个非 Loss 输出)
#             likelihoods = outputs[0] # 形状 [B, 5]，即 P(Y=k)
            
#             # 4. 计算最终预测分数
            
#             # 平均预测值（期望值）：Sum(P(Y=k) * k)
#             mean_predictions = (likelihoods * score_values).sum(dim=1) 
            
#             # 四舍五入到最近的整数 (1-5)
#             final_scores_float = mean_predictions.cpu().numpy()
#             final_scores = np.round(final_scores_float).astype(int)
            
#             # 确保分数在 1 到 5 之间
#             final_scores = np.clip(final_scores, 1, 5) 
            
#             # 5. 收集结果
#             for idx, pred_score in enumerate(final_scores):
#                 all_results.append({
#                     "id": ids[idx],
#                     "prediction": int(pred_score),
#                     "mean_prediction": final_scores_float[idx],
#                     # 如果需要，可以保存完整的似然分布: likelihoods[idx].tolist()
#                 })
                
#     # 6. 保存结果到文件
#     output_filepath = "/kaggle/working/predictions.json"
#     with open(output_filepath, 'w', encoding='utf-8') as f:
#         json.dump(all_results, f, ensure_ascii=False, indent=4)
        
#     print(f"\n✅ 推理完成，结果已保存到 {output_filepath}")

# if __name__ == '__main__':
#     # 确保在运行前，InferenceDataset 和 DebertaV2ForOrdinalRegression 已定义或导入
#     # 运行主推理函数
#     run_inference()

In [10]:
# # 检查当前GPU使用情况
# !nvidia-smi

# # 查看Python进程使用的GPU内存
# import pynvml
# pynvml.nvmlInit()
# handle = pynvml.nvmlDeviceGetHandleByIndex(0)
# info = pynvml.nvmlDeviceGetMemoryInfo(handle)
# print(f"GPU内存使用情况: {info.used//1024**2}MB / {info.total//1024**2}MB")